# Lab 1: Crawling and crawl scheduling

**Goal:** scrape 50 product pages, re-crawl them on a schedule while prices change underneath you, and compare two scheduling policies:

- **Policy A, uniform:** re-crawl every item once a day.
- **Policy B, adaptive:** re-crawl items that changed recently more often, and back off on items that don't.

**Takeaway question:** *how much staleness does each policy leave per unit of crawl cost?*

**How this notebook works**
1. **Part 1** scrapes real pages from [books.toscrape.com](https://books.toscrape.com), a site built for scraping practice.
2. **Part 2** wraps those pages in a simulated site whose prices change over 30 days. The real site never changes, so we need this to have anything to re-crawl.
3. **Parts 3–5** run the policies against the *same* simulated world, so the comparison is fair.

Cells marked **`# TODO`** are yours to fill in. Each one has a self-check cell right after it. Budget: one evening.

## Setup

In [ ]:
# Everything here is preinstalled on Colab.
import re, time, random
from urllib.parse import urljoin
from urllib import robotparser

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = "https://books.toscrape.com/"
USER_AGENT = "lab1-crawler (learning project)"
N_ITEMS = 50
POLITE_DELAY_S = 0.3   # pause between requests; be a good citizen

---
## Part 1: Crawl 50 product pages

A crawler has three jobs: **discover** URLs, **fetch** pages politely, and **parse** them into structured records.
Discovery and fetching are written for you. Parsing is your first TODO.

In [ ]:
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

def allowed_by_robots(url):
    # Check robots.txt before crawling. books.toscrape.com has none, which means everything is allowed.
    rp = robotparser.RobotFileParser(urljoin(BASE, "robots.txt"))
    try:
        rp.read()
    except Exception:
        return True
    return rp.can_fetch(USER_AGENT, url)

def fetch(url, retries=3):
    # GET with a timeout, retries with exponential backoff, and a polite delay.
    for attempt in range(retries):
        try:
            r = session.get(url, timeout=10)
            r.raise_for_status()
            r.encoding = "utf-8"   # the site is UTF-8; without this '£' decodes as 'Â£'
            time.sleep(POLITE_DELAY_S)
            return r.text
        except requests.RequestException as e:
            wait = 2 ** attempt
            print(f"  fetch failed ({e.__class__.__name__}); retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"giving up on {url}")

def discover_product_urls(n):
    # Walk the paginated catalogue, collecting product links until we have n.
    urls, page_url = [], urljoin(BASE, "catalogue/page-1.html")
    while page_url and len(urls) < n:
        soup = BeautifulSoup(fetch(page_url), "html.parser")
        urls += [urljoin(page_url, a["href"]) for a in soup.select("article.product_pod h3 a")]
        nxt = soup.select_one("li.next a")
        page_url = urljoin(page_url, nxt["href"]) if nxt else None
    return urls[:n]

### TODO 1: parse a product page

Open any product page, e.g. [A Light in the Attic](https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html), right-click and choose **Inspect**. You'll find:

| Field | Where it lives |
|---|---|
| `title` | `div.product_main h1` |
| `price` | `div.product_main p.price_color` → text like `£51.77`. Convert it to a `float`. |
| `stock` | `div.product_main p.availability` → `In stock (22 available)`. Extract the int, or 0 if missing. |
| `rating` | `p.star-rating` has a second class like `Three`. Map it to `3`. |
| `upc` | the `table.table-striped` row whose `<th>` is `UPC`. This is the stable item ID. |
| `category` | the last link in `ul.breadcrumb` |

`soup.select_one(css)`, `.get_text(strip=True)`, `tag["class"]` and `re.search` are all you need.

In [ ]:
RATING_WORDS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def parse_product_page(html, url=None):
    soup = BeautifulSoup(html, "html.parser")
    # TODO: extract the fields described above and return them as a dict:
    # {"upc": str, "title": str, "price": float, "stock": int,
    #  "rating": int, "category": str, "url": url}
    raise NotImplementedError("TODO 1: parse_product_page")

### Self-check for TODO 1
This runs your parser on a small page with the same structure as the real site, so you can debug without hitting the network.

In [ ]:
def render_product_html(title, price, stock, rating, upc, category):
    # Minimal page with the same markup as a books.toscrape.com product page.
    # Also used later by the simulated site, and as an offline fallback.
    rating_word = {v: k for k, v in RATING_WORDS.items()}[rating]
    return f'''<html><body>
<ul class="breadcrumb"><li><a href="../../index.html">Home</a></li>
<li><a href="../category/books_1/index.html">Books</a></li>
<li><a href="../category/books/x_1/index.html">{category}</a></li><li class="active">{title}</li></ul>
<div class="row"><div class="col-sm-6 product_main"><h1>{title}</h1>
<p class="price_color">£{price:.2f}</p>
<p class="instock availability"><i class="icon-ok"></i> In stock ({stock} available)</p>
<p class="star-rating {rating_word}"><i class="icon-star"></i></p></div></div>
<table class="table table-striped"><tr><th>UPC</th><td>{upc}</td></tr>
<tr><th>Product Type</th><td>Books</td></tr><tr><th>Price (excl. tax)</th><td>£{price:.2f}</td></tr></table>
</body></html>'''

sample = render_product_html("A Light in the Attic", 51.77, 22, 3, "a897fe39b1053632", "Poetry")
got = parse_product_page(sample, "u")
expected = {"upc": "a897fe39b1053632", "title": "A Light in the Attic", "price": 51.77,
            "stock": 22, "rating": 3, "category": "Poetry", "url": "u"}
for k, v in expected.items():
    assert got.get(k) == v, f"{k}: expected {v!r}, got {got.get(k)!r}"
print("✅ parser looks good:", got)

### Run the crawl
Takes about 30 seconds (50 pages plus 3 catalogue pages, with a polite delay). If the site is unreachable, the cell falls back to a synthetic catalogue so the rest of the lab still works.

In [ ]:
pages = {}   # upc -> raw HTML; the simulated site in Part 2 serves these back
try:
    assert allowed_by_robots(urljoin(BASE, "catalogue/")), "robots.txt disallows crawling"
    urls = discover_product_urls(N_ITEMS)
    records = []
    for i, url in enumerate(urls, 1):
        html = fetch(url)
        rec = parse_product_page(html, url)
        pages[rec["upc"]] = html
        records.append(rec)
        if i % 10 == 0:
            print(f"crawled {i}/{len(urls)}")
    DATA_SOURCE = "books.toscrape.com"
except Exception as e:
    print(f"⚠️ Live crawl failed ({e}). Using a synthetic catalogue instead.")
    rng = np.random.default_rng(SEED)
    records = []
    for i in range(N_ITEMS):
        rec = dict(upc=f"synthetic{i:04d}", title=f"Book {i}", price=round(float(rng.uniform(10, 60)), 2),
                   stock=int(rng.integers(1, 30)), rating=int(rng.integers(1, 6)), category="Fiction")
        pages[rec["upc"]] = render_product_html(**rec)
        records.append({**rec, "url": None})
    DATA_SOURCE = "synthetic"

catalog = pd.DataFrame(records)
catalog.to_csv("products_day0.csv", index=False)
print(f"\n{len(catalog)} items from {DATA_SOURCE}")
catalog.head()

In [ ]:
# Sanity checks: always validate a crawl before trusting it.
assert len(catalog) == N_ITEMS, "expected 50 items"
assert catalog["upc"].is_unique, "duplicate UPCs: are you parsing the right table row?"
assert (catalog["price"] > 0).all(), "some prices failed to parse"
catalog.describe()

---
## Part 2: A site that changes over time

books.toscrape.com is static, so we simulate 30 days of price changes **hour by hour**. Each item gets a volatility tier, as in real catalogs where a few items churn constantly and most rarely change:

| Tier | Share of items | Average price changes |
|---|---|---|
| hot | 10% | 2 per day |
| warm | 20% | 1 every 3 days |
| cold | 70% | 1 per month |

Changes happen at random hours, so even a daily crawler is a few hours behind. Every policy runs against the **same** precomputed world. `SimulatedSite.fetch()` returns real page HTML with the price swapped in, so your parser does the actual work on every re-crawl.

In [ ]:
DAYS = 30
HOURS = 24 * DAYS
TIERS = {             # tier: (share of items, changes per day)
    "hot":  (0.10, 2.0),
    "warm": (0.20, 1 / 3),
    "cold": (0.70, 1 / 30),
}

def simulate_world(base_prices, hours=HOURS, seed=SEED):
    # Returns true_prices[item, hour], each item's tier, and change_events[item, hour].
    rng = np.random.default_rng(seed)
    n = len(base_prices)
    tiers = rng.choice(list(TIERS), size=n, p=[share for share, _ in TIERS.values()])
    hourly_rate = np.array([TIERS[t][1] / 24 for t in tiers])
    change_events = rng.random((n, hours)) < hourly_rate[:, None]
    change_events[:, 0] = False                       # hour 0 is the initial crawl
    moves = np.where(change_events, rng.uniform(0.85, 1.15, (n, hours)), 1.0)
    true_prices = np.round(np.asarray(base_prices)[:, None] * np.cumprod(moves, axis=1), 2)
    return true_prices, tiers, change_events

class SimulatedSite:
    # Serves product HTML as it looks at a given hour. Every fetch costs one crawl.
    PRICE_RE = re.compile(r'(<p class="price_color">)[^<]*(</p>)')

    def __init__(self, upcs, pages, true_prices):
        self.upcs, self.pages, self.true_prices = list(upcs), pages, true_prices
        self.fetches = 0

    def fetch(self, item, hour):
        self.fetches += 1
        price = self.true_prices[item, hour]
        return self.PRICE_RE.sub(rf"\g<1>£{price:.2f}\g<2>", self.pages[self.upcs[item]], count=1)

true_prices, tiers, change_events = simulate_world(catalog["price"].values)
catalog["tier"] = tiers
print(pd.Series(tiers).value_counts().to_string())
print(f"\nTotal price changes over {DAYS} days: {change_events.sum()}")
print("By tier:", {t: int(change_events[tiers == t].sum()) for t in TIERS})

In [ ]:
# Quick look at the world: a few items' true price over time.
fig, ax = plt.subplots(figsize=(10, 3.5))
for tier in TIERS:
    i = int(np.flatnonzero(tiers == tier)[0])
    ax.step(np.arange(HOURS) / 24, true_prices[i], where="post", label=f"{tier}: {catalog.title[i][:30]}")
ax.set(xlabel="day", ylabel="true price (£)", title="One item per tier")
ax.legend(fontsize=8); plt.show()

---
## Part 3: The crawl scheduler

A **policy** answers one question every simulated hour: *which items should I re-crawl now?* It does this through two methods:
- `due(hour)` returns a boolean mask of items to crawl this hour.
- `observe(item, hour, changed)` reports what a crawl found, so the policy can adapt.

`run_policy` is the harness. It keeps our copy of the catalog (`known_prices`), fetches and parses whatever is due, and measures **staleness** every hour: how many items show a price that no longer matches the site.

Policy A is written for you.

In [ ]:
class FixedInterval:
    # Policy A: re-crawl every item every `every_h` hours, no matter what.
    def __init__(self, every_h=24):
        self.every_h = every_h
        self.name = f"fixed every {every_h}h"

    def reset(self, n):
        self.next_due = np.full(n, self.every_h)

    def due(self, hour):
        return self.next_due <= hour

    def observe(self, item, hour, changed):
        self.next_due[item] = hour + self.every_h


def run_policy(policy, site, hours=HOURS):
    n = len(site.upcs)
    known_prices = site.true_prices[:, 0].copy()   # initial full crawl at hour 0 (identical for every policy, so not counted)
    policy.reset(n)
    site.fetches = 0
    stale_per_hour = np.zeros(hours, dtype=int)
    stale_item_hours = np.zeros(n, dtype=int)       # per item, for the per-tier breakdown
    found_change = 0
    for hour in range(hours):
        for item in np.flatnonzero(policy.due(hour)):
            rec = parse_product_page(site.fetch(item, hour))
            changed = rec["price"] != known_prices[item]
            known_prices[item] = rec["price"]
            found_change += changed
            policy.observe(item, hour, changed)
        stale = known_prices != site.true_prices[:, hour]
        stale_per_hour[hour] = stale.sum()
        stale_item_hours += stale
    return {"policy": policy.name, "crawls": site.fetches, "crawls_finding_change": found_change,
            "stale_per_hour": stale_per_hour, "stale_item_hours": stale_item_hours}

### TODO 2: the adaptive policy (Policy B)

Give each item its own re-crawl interval and adjust it after every crawl:
- **changed** → crawl sooner: `interval *= speedup` (e.g. 0.5), but never below `min_h`
- **unchanged** → back off: `interval *= backoff` (e.g. 1.2), but never above `max_h`
- then schedule the next crawl: `next_due[item] = hour + round(interval)`

This is the same idea as TCP backoff, and close to what production crawlers do before adding change-rate models.

In [ ]:
class Adaptive:
    # Policy B: crawl items that change often more often.
    def __init__(self, start_h=24, min_h=4, max_h=96, speedup=0.5, backoff=1.2):
        self.start_h, self.min_h, self.max_h = start_h, min_h, max_h
        self.speedup, self.backoff = speedup, backoff
        self.name = f"adaptive {min_h}-{max_h}h"

    def reset(self, n):
        self.interval = np.full(n, float(self.start_h))
        self.next_due = np.full(n, self.start_h)

    def due(self, hour):
        return self.next_due <= hour

    def observe(self, item, hour, changed):
        # TODO 2: update self.interval[item], then self.next_due[item]
        raise NotImplementedError("TODO 2: Adaptive.observe")

In [ ]:
# Self-check for TODO 2
p = Adaptive(start_h=24, min_h=4, max_h=96, speedup=0.5, backoff=1.2); p.reset(2)
p.observe(0, 24, changed=True);  assert p.interval[0] == 12 and p.next_due[0] == 36, "changed should halve the interval"
p.observe(1, 24, changed=False); assert np.isclose(p.interval[1], 28.8) and p.next_due[1] == 53, "unchanged should multiply by 1.2"
for _ in range(10): p.observe(0, 0, True)
assert p.interval[0] == 4, "interval must not drop below min_h"
for _ in range(30): p.observe(1, 0, False)
assert p.interval[1] == 96, "interval must not exceed max_h"
print("✅ adaptive policy looks good")

---
## Part 4: Head to head

### TODO 3: summarize a run
Turn the raw output of `run_policy` into the numbers you'd put on a slide:
- `crawls`: total fetches (the **cost**)
- `avg_pct_stale`: average share of the catalog that is stale at any moment (the **quality**)
- `stale_item_hours`: total item-hours spent stale (sum of `stale_item_hours`)
- `wasted_crawl_pct`: share of crawls that found no change
- `stale_hours_per_100_crawls`: **staleness per unit of cost**. Lower is better, but think about what it means when the crawl count is tiny.

In [ ]:
def summarize(result, n_items=N_ITEMS):
    # TODO 3: return a dict with: policy, crawls, avg_pct_stale, stale_item_hours,
    #         wasted_crawl_pct, stale_hours_per_100_crawls
    raise NotImplementedError("TODO 3: summarize")

In [ ]:
# Self-check for TODO 3
fake = {"policy": "x", "crawls": 200, "crawls_finding_change": 50,
        "stale_per_hour": np.array([5, 5, 10, 0]), "stale_item_hours": np.array([10, 10])}
s = summarize(fake, n_items=50)
assert s["crawls"] == 200 and s["stale_item_hours"] == 20
assert s["avg_pct_stale"] == 10.0, s["avg_pct_stale"]
assert s["wasted_crawl_pct"] == 75.0, s["wasted_crawl_pct"]
assert s["stale_hours_per_100_crawls"] == 10.0, s["stale_hours_per_100_crawls"]
print("✅ summarize looks good")

In [ ]:
site = SimulatedSite(catalog["upc"], pages, true_prices)
res_a = run_policy(FixedInterval(24), site)
res_b = run_policy(Adaptive(), site)
pd.DataFrame([summarize(res_a), summarize(res_b)]).set_index("policy")

In [ ]:
# Where does the staleness come from? Break it down by tier.
def by_tier(result):
    return pd.Series(result["stale_item_hours"]).groupby(tiers).sum()

breakdown = pd.DataFrame({res_a["policy"]: by_tier(res_a), res_b["policy"]: by_tier(res_b)})
breakdown.loc["TOTAL"] = breakdown.sum()
print("Stale item-hours by tier")
breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
for r in (res_a, res_b):
    ax.plot(np.arange(HOURS) / 24, 100 * r["stale_per_hour"] / N_ITEMS, label=r["policy"], lw=1)
ax.set(xlabel="day", ylabel="% of catalog stale", title="Staleness over time")
ax.legend(); plt.show()

---
## Part 5: The cost–freshness frontier

One run of each policy is just one point. Sweep the knobs to trace each policy's **curve**: crawl cost on the x-axis, staleness on the y-axis. Better policies sit **down and to the left**. At any budget, compare the curves vertically.

This takes about a minute because every crawl really parses HTML.

In [ ]:
sweep = [FixedInterval(h) for h in (6, 12, 24, 48, 96)] + [
    Adaptive(min_h=lo, max_h=hi) for lo, hi in ((1, 48), (2, 72), (4, 96), (8, 168), (16, 336))
]
frontier = pd.DataFrame([summarize(run_policy(p, site)) for p in sweep])
frontier["family"] = np.where(frontier.policy.str.startswith("fixed"), "A: fixed", "B: adaptive")

fig, ax = plt.subplots(figsize=(8, 5))
for fam, g in frontier.groupby("family"):
    g = g.sort_values("crawls")
    ax.plot(g.crawls, g.avg_pct_stale, "o-", label=fam)
    for _, r in g.iterrows():
        ax.annotate(r.policy.split(" ", 1)[1], (r.crawls, r.avg_pct_stale), fontsize=7,
                    xytext=(4, 4), textcoords="offset points")
ax.set(xscale="log", xlabel="crawls over 30 days (cost, log scale)", ylabel="avg % of catalog stale",
       title="Cost vs. freshness")
ax.legend(); plt.show()
frontier.drop(columns="family").set_index("policy")

---
## Part 6: Write-up (do this, it's the interview material)

Answer the takeaway question with **your numbers**: *how much staleness does each policy leave per unit of crawl cost?*

Then write three sentences:
1. **The tradeoff I saw:**
2. **The number that moved:** (e.g. "same staleness at X% fewer crawls" or "Y% less staleness at the same cost")
3. **How I'd explain it to an executive:**

**Questions to think about**
- Adaptive probably won by less than you expected. Why? Think about the hot items (can you ever keep up with something that changes faster than you crawl?) and the cold items (how long does a rare change go unnoticed once you've backed off?).
- Try the textbook knobs, `Adaptive(max_h=168, backoff=1.5)`. Where does that point land on the frontier, and why?
- Look at the per-tier table. Which tier causes most of Policy A's staleness? Why can't a uniform schedule fix that without crawling everything more often?
- What does adaptive do to the **cold** items? What's the risk if a cold item suddenly gets hot, like a price drop on Black Friday?
- Why is `stale_hours_per_100_crawls` misleading on its own? (Hint: what happens to it for a policy that almost never crawls?)
- Real crawlers have a fixed **budget** (crawls per hour, set by politeness limits and compute). How would you change `Adaptive` to respect a hard budget?

**Stretch goals**
- Change `TIERS` so 50% of items are hot. Does adaptive still win? By how much?
- Weight staleness by **importance**: a stale price on a bestseller hurts more. Use `rating` or a fake popularity score as the weight.
- Replace halve/double with a change-rate estimate: `changes_seen / hours_observed` per item, and crawl proportional to √rate (Cho & Garcia-Molina's result on optimal refresh).
- Read about HTTP `ETag` / `If-Modified-Since`. How would conditional requests change the cost model?

**Write your answers here:**

1. Tradeoff:
2. Number:
3. Executive version: